# Part II · Lesson 2 — Choose the path: food delivery

## Our state and its update rules

### Import the graph and update helpers

In [1]:
from operator import add
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Overwrite

### Merge separate check results

In [2]:
def merge_checks(current: dict, update: dict) -> dict:
    return {**current, **update}

### Keep the larger attempt count

In [3]:
def keep_larger(current: int, update: int) -> int:
    return max(current, update)

### Describe the delivery state

In [4]:
class DeliveryState(TypedDict):
    order: dict
    checks: Annotated[dict[str, bool], merge_checks]
    attempts: Annotated[int, keep_larger]
    status: str
    logs: Annotated[list[str], add]

### Prepare sample input

In [5]:
sample_order = {
    "order": {
        "restaurant": "Spice Kitchen", "address": " 12 Park Road ",
        "preferred_rider": "Asha", "rider_responses": [False, True, False],
    },
    "checks": {}, "attempts": 0, "status": "new", "logs": [],
}

## 1. Sequential — do this, then that

### Read and validate the order

In [6]:
def read_order(state: DeliveryState) -> dict:
    order = {**state["order"], "address": state["order"]["address"].strip()}
    return {"order": order, "logs": ["Order read."]}


def validate_address(state: DeliveryState) -> dict:
    if not state["order"]["address"]:
        raise ValueError("A delivery address is required.")
    return {"status": "validated", "logs": ["Address validated."]}

### Connect and run the sequential graph

In [7]:
sequential_builder = StateGraph(DeliveryState)
sequential_builder.add_node("read", read_order)
sequential_builder.add_node("validate", validate_address)
sequential_builder.add_edge(START, "read")
sequential_builder.add_edge("read", "validate")
sequential_builder.add_edge("validate", END)
sequential_result = sequential_builder.compile().invoke(sample_order)
print(sequential_result)
prepared_order = {**sequential_result, "logs": []}

{'order': {'restaurant': 'Spice Kitchen', 'address': '12 Park Road', 'preferred_rider': 'Asha', 'rider_responses': [False, True, False]}, 'checks': {}, 'attempts': 0, 'status': 'validated', 'logs': ['Order read.', 'Address validated.']}


## 2. Parallel — independent nodes which both needs to run

### Define the two nodes

In [8]:
def check_restaurant(state: DeliveryState) -> dict:
    restaurants = {"Spice Kitchen": True, "Night Cafe": False}
    is_open = restaurants.get(state["order"]["restaurant"], False)
    return {"checks": {"restaurant": is_open}, "logs": ["Restaurant checked."]}


def check_rider(state: DeliveryState) -> dict:
    riders = {"Asha": True, "Ravi": False}
    available = riders.get(state["order"]["preferred_rider"], False)
    return {"checks": {"rider": available}, "logs": ["Rider checked."]}

### Define the step after both checks

In [9]:
def finish_checks(state: DeliveryState) -> dict:
    return {"status": "checks complete", "logs": ["Both checks finished."]}

### Split the path and join the results

In [10]:
parallel_builder = StateGraph(DeliveryState)
parallel_builder.add_node("restaurant", check_restaurant)
parallel_builder.add_node("rider", check_rider)
parallel_builder.add_node("finish", finish_checks)
parallel_builder.add_edge(START, "restaurant")
parallel_builder.add_edge(START, "rider")
# Wait for both checks before running finish.
parallel_builder.add_edge(["restaurant", "rider"], "finish")
parallel_builder.add_edge("finish", END)
parallel_graph = parallel_builder.compile()

### Inspect the parallel result

In [11]:
parallel_result = parallel_graph.invoke(prepared_order)
print(parallel_result["checks"])
print(parallel_result["logs"])

{'restaurant': True, 'rider': True}
['Restaurant checked.', 'Rider checked.', 'Both checks finished.']


## Partial versus full returns — why parallel work makes this matter

### Show a full return that resends an old log

In [12]:
def full_status_update(state: DeliveryState) -> dict:
    return {**state, "status": "reviewed"}

full_builder = StateGraph(DeliveryState)
full_builder.add_node("review", full_status_update)
full_builder.add_edge(START, "review")
full_builder.add_edge("review", END)
full_result = full_builder.compile().invoke(
    {**prepared_order, "logs": ["Existing entry."]}
)
print(full_result["logs"])

['Existing entry.', 'Existing entry.']


### Define the unsafe parallel versions

In [13]:
def full_restaurant_check(state: DeliveryState) -> dict:
    return {**state, **check_restaurant(state)}


def full_rider_check(state: DeliveryState) -> dict:
    return {**state, **check_rider(state)}

### Build the demonstration of conflicting writes

In [14]:
unsafe_builder = StateGraph(DeliveryState)
unsafe_builder.add_node("restaurant", full_restaurant_check)
unsafe_builder.add_node("rider", full_rider_check)
unsafe_builder.add_edge(START, "restaurant")
unsafe_builder.add_edge(START, "rider")
unsafe_builder.add_edge("restaurant", END)
unsafe_builder.add_edge("rider", END)
unsafe_graph = unsafe_builder.compile()
# Optional: uncomment to see InvalidUpdateError for simultaneous full returns.
# unsafe_graph.invoke(prepared_order)

## 3. Conditional — choose one next step

### Define the two branch outcomes

In [15]:
def find_rider(state: DeliveryState) -> dict:
    result = check_rider(state)
    result["status"] = "rider found" if result["checks"]["rider"] else "rider needed"
    return result


def reject_order(state: DeliveryState) -> dict:
    return {"status": "restaurant closed", "logs": ["Order rejected."]}

### Choose the open or closed route

In [16]:
def restaurant_route(state: DeliveryState) -> str:
    if state["checks"]["restaurant"]:
        return "open"
    return "closed"

### Connect the conditional graph

In [17]:
conditional_builder = StateGraph(DeliveryState)
conditional_builder.add_node("restaurant", check_restaurant)
conditional_builder.add_node("find_rider", find_rider)
conditional_builder.add_node("reject_order", reject_order)
conditional_builder.add_edge(START, "restaurant")
conditional_builder.add_conditional_edges(
    "restaurant", restaurant_route,
    {"open": "find_rider", "closed": "reject_order"},
)
conditional_builder.add_edge("find_rider", END)
conditional_builder.add_edge("reject_order", END)
conditional_graph = conditional_builder.compile()

### Try both restaurant cases

In [18]:
open_result = conditional_graph.invoke(prepared_order)
closed_order = {**prepared_order["order"], "restaurant": "Night Cafe"}
closed_result = conditional_graph.invoke(
    {**prepared_order, "order": closed_order}
)
print("Open:", open_result["status"], open_result["logs"])
print("Closed:", closed_result["status"], closed_result["logs"])

Open: rider found ['Restaurant checked.', 'Rider checked.']
Closed: restaurant closed ['Restaurant checked.', 'Order rejected.']


## 4. Bounded loop — another attempt, with an exit

### Try the next candidate

In [19]:
def search_rider(state: DeliveryState) -> dict:
    attempt = state["attempts"] + 1
    responses = state["order"]["rider_responses"]
    available = responses[attempt - 1] if attempt <= len(responses) else False
    return {
        "attempts": attempt, "checks": {"rider": available},
        "logs": [f"Rider attempt {attempt}: {available}"],
    }

### Decide whether to repeat or finish

In [20]:
def rider_route(state: DeliveryState) -> str:
    if state["checks"]["rider"]:
        return "assigned"
    if state["attempts"] >= 3:
        return "unavailable"
    return "again"


def confirm_dispatch(state: DeliveryState) -> dict:
    return {"status": "confirmed", "logs": ["Delivery confirmed."]}


def no_rider_available(state: DeliveryState) -> dict:
    return {"status": "no rider available", "logs": ["Search stopped."]}

### Connect the loop

In [21]:
loop_builder = StateGraph(DeliveryState)
loop_builder.add_node("search", search_rider)
loop_builder.add_node("confirm", confirm_dispatch)
loop_builder.add_node("no_rider", no_rider_available)
loop_builder.add_edge(START, "search")
loop_builder.add_conditional_edges(
    "search", rider_route,
    {"again": "search", "assigned": "confirm", "unavailable": "no_rider"},
)
loop_builder.add_edge("confirm", END)
loop_builder.add_edge("no_rider", END)
loop_graph = loop_builder.compile()

### Run success and exhaustion cases

In [22]:
loop_result = loop_graph.invoke(prepared_order, {"recursion_limit": 12})
failed_order = {**prepared_order["order"], "rider_responses": [False, False, False]}
exhausted_result = loop_graph.invoke(
    {**prepared_order, "order": failed_order}, {"recursion_limit": 12}
)
print("Accepted:", loop_result["attempts"], loop_result["status"])
print("Exhausted:", exhausted_result["attempts"], exhausted_result["status"])

Accepted: 2 confirmed
Exhausted: 3 no rider available


## Put the pieces together

### Choose the combined next step

In [23]:
def dispatch_route(state: DeliveryState) -> str:
    if not state["checks"]["restaurant"]:
        return "closed"
    if state["checks"]["rider"]:
        return "ready"
    return "search"

### Register the combined workflow

In [24]:
dispatch_builder = StateGraph(DeliveryState)
for name, function in [
    ("read", read_order), ("validate", validate_address),
    ("restaurant", check_restaurant), ("rider", check_rider),
    ("finish", finish_checks), ("reject", reject_order),
    ("search", search_rider), ("confirm", confirm_dispatch),
    ("no_rider", no_rider_available),
]:
    dispatch_builder.add_node(name, function)

### Connect preparation, parallel checks, and the decision

In [25]:
dispatch_builder.add_edge(START, "read")
dispatch_builder.add_edge("read", "validate")
dispatch_builder.add_edge("validate", "restaurant")
dispatch_builder.add_edge("validate", "rider")
dispatch_builder.add_edge(["restaurant", "rider"], "finish")
dispatch_builder.add_conditional_edges(
    "finish", dispatch_route,
    {"closed": "reject", "ready": "confirm", "search": "search"},
)

### Attach the bounded search and run the workflow

In [26]:
dispatch_builder.add_conditional_edges(
    "search", rider_route,
    {"again": "search", "assigned": "confirm", "unavailable": "no_rider"},
)
for final_node in ["reject", "confirm", "no_rider"]:
    dispatch_builder.add_edge(final_node, END)
dispatch_graph = dispatch_builder.compile()
dispatch_result = dispatch_graph.invoke(sample_order, {"recursion_limit": 20})
print(dispatch_result["status"], dispatch_result["logs"])

confirmed ['Order read.', 'Address validated.', 'Restaurant checked.', 'Rider checked.', 'Both checks finished.', 'Delivery confirmed.']


## Resetting an accumulated field deliberately

### Clear the log with an explicit overwrite

In [27]:
def clear_log(state: DeliveryState) -> dict:
    return {"logs": Overwrite(value=[])}

reset_builder = StateGraph(DeliveryState)
reset_builder.add_node("clear", clear_log)
reset_builder.add_edge(START, "clear")
reset_builder.add_edge("clear", END)
reset_result = reset_builder.compile().invoke(parallel_result)
print(reset_result["logs"])

[]


## The same reducer can mean different things

### Compare adding a delta with keeping a total

In [28]:
# A sum-based counter receives a delta: one additional attempt.
print(add(2, 1))  # 3
# Our counter receives the new total: three attempts so far.
print(keep_larger(2, 3))  # 3
print(keep_larger(3, 1))  # 3: a smaller update cannot lower it

3
3
3
